In [ ]:
import random
import io
import os
import json
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

# 1. Lista de 90 conceptos y fórmulas
conceptos = [
    r"$\pi$", r"$e$", r"$i$", r"$\phi$", r"$\aleph_0$", r"$\gamma$", r"$\infty$",
    r"$e^{i\pi} + 1 = 0$", "Número Real", "15", r"$2x$", r"$e^x$", r"$\cos(x)$", r"$-\sin(x)$", r"$\frac{1}{x}$",
    "Matriz Diagonal", r"$15x^2$", r"$6x$", "16", r"$2\pi$", "Teorema de\nBolzano-Weierstrass",
    "Teorema de\nHeine-Borel", "Teorema del\nValor Medio", "Base", "Dimensión", "Núcleo\n(Kernel)",
    "Proceso de\nGram-Schmidt", "Integral de\nLebesgue", "Matriz\nJacobiana", "Gradiente", "Supremo",
    "Sucesión de\nCauchy", r"$\frac{1}{\sqrt{1 - x^2}}$", r"$(1 - \lambda)^2$", "-2", "36", "0", "(3, 4)",
    "Vector columna", "Teorema de\nStokes", "Teorema de\nGreen", "4!", "Divergencia", "Laplaciano",
    "Teorema de\nBayes", "Distribución\nNormal", "Teorema del\nLímite Central", "45°", "2", "Grafo", "1",
    "Teoría del\nCaos", "6", "5", "Teorema de los\nCuatro Colores", "120", "Correlación de\nPearson",
    "Distribución de\nPoisson", "Rango\nestadístico", "Grupo", "Grupo\nAbeliano", "Cuerpo",
    "Homomorfismo", "Teorema de\nLagrange", "Grupo Simple", "33", "Criptografía\nRSA", "Conjunto de\nMandelbrot",
    "5050", "32", "3", "-7", "Homeomorfismo", "Banda de\nMöbius", "Función Entera", "Residuo",
    "Teorema de\nIncompletitud", "Hipótesis del\nContinuo", "Paradoja de\nRussell", "Axioma de\nElección",
    "Desigualdad de\nCauchy-Schwarz", "Hipótesis de\nRiemann", "Matriz Definida\nPositiva", "Fractal",
    "Cuerno de\nGabriel", r"$-1$", "Álgebra de\nBoole", "Reducción al\nAbsurdo", "Movimiento\nBrowniano",
    "Base Canónica"
]

# 2. Coordenadas detectadas de la cuadrícula en tu plantilla
row_peaks = [2121, 2821, 3569, 4341, 5041, 5741]
col_peaks = [451, 1163, 1878, 2583, 3300, 4012]

# Función para generar la imagen del texto/ecuación
def render_text_to_image(text, width, height):
    fig = plt.figure(figsize=(width/100, height/100), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis('off')

    # 1. Identificar si el texto es una fórmula matemática (contiene $)
    es_formula = "$" in text

    if es_formula:
        # Reemplazamos \frac por \dfrac para que la fracción se vea grande y legible
        if "\\frac" in text:
            text = text.replace("\\frac", "\\dfrac")

        fs = 75 # Tamaño gigante para fórmulas
        ajustar_texto = False # No romper fórmulas
    else:
        # --- EXCEPCIONES PARA PALABRAS ESPECÍFICAS ---
        if text in ["Homomorfismo", "Homeomorfismo"]:
            fs = 45  # Forzamos un tamaño más pequeño para que quepan en una línea
        elif "Bolzano-Weierstrass" in text:
            fs = 40  # Aún más pequeño porque es muy largo
        # ---------------------------------------------
        else:
            # Tamaños dinámicos para los demás textos normales
            fs = 65
            if len(text) > 20: fs = 50
            if len(text) > 30: fs = 40

        ajustar_texto = True # Romper textos normales largos

    ax.text(0.5, 0.5, text, fontsize=fs, ha='center', va='center', wrap=ajustar_texto, color='#69314c', weight='bold')

    buf = io.BytesIO()
    plt.savefig(buf, format='png', transparent=True)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf)


# 3. Generar los 80 cartones y guardar sus matrices correspondientes
print("Cargando plantilla...")
plantilla_original = Image.open("Carton bingo.png").convert("RGBA")
cartones_generados = []
registro_cartones = {}

cantidad_cartones = 80

print(f"Generando {cantidad_cartones} cartones. Esto puede tomar un par de minutos...")
for n in range(cantidad_cartones):
    # Crear una copia fresca de la plantilla
    carton = plantilla_original.copy()

    # Seleccionar 24 conceptos al azar para este cartón
    seleccion = random.sample(conceptos, 24)

    # Almacenamos la lista de conceptos del cartón en el registro (indexado desde el cartón 1 al 80)
    registro_cartones[f"carton_{n+1}"] = [concept.replace('\n', ' ') for concept in seleccion]

    idx = 0
    for j in range(5):
        for i in range(5):
            # Saltar la casilla central (Espacio Libre)
            if i == 2 and j == 2:
                continue

            x_min, x_max = col_peaks[i], col_peaks[i+1]
            y_min, y_max = row_peaks[j], row_peaks[j+1]

            cell_w = x_max - x_min
            cell_h = y_max - y_min

            # Renderizar y pegar el texto
            text_img = render_text_to_image(seleccion[idx], cell_w, cell_h)
            idx += 1
            carton.alpha_composite(text_img, (x_min, y_min))
        # --- AGREGAR EL NÚMERO DE CARTÓN ---
    draw = ImageDraw.Draw(carton)
    texto_id = f"Cartón #{n + 1}"

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 60)
    except IOError:
        font = ImageFont.load_default()

    draw.text((2000, 5750), texto_id, fill='#69314c', font=font)

    carton_rgb = carton.convert("RGB")
    carton_rgb.thumbnail((2232, 3125))
    cartones_generados.append(carton_rgb)

    if (n + 1) % 10 == 0:
        print(f"Completados {n + 1} de {cantidad_cartones}...")

# 4. Guardar todo en un único archivo PDF
pdf_path = "80_Cartones_Math_Bingo.pdf"
cartones_generados[0].save(
    pdf_path,
    save_all=True,
    append_images=cartones_generados[1:]
)

# Guardar la estructura de datos en un archivo JSON
json_path = "cartones_data.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(registro_cartones, f, ensure_ascii=False, indent=4)

print(f"¡Completado! Matrices guardadas en {json_path}")

Cargando plantilla...
Generando 80 cartones. Esto puede tomar un par de minutos...
Completados 10 de 80...
Completados 20 de 80...
Completados 30 de 80...
Completados 40 de 80...
Completados 50 de 80...
Completados 60 de 80...
Completados 70 de 80...
Completados 80 de 80...
¡Completado! Matrices guardadas en cartones_data.json


In [ ]:
# 5. Descargar los archivos generados
from google.colab import files

print("¡Generación completada! Iniciando descarga de archivos...")

# Descargar el PDF
print("Descargando PDF...")
files.download(pdf_path)

# Descargar el JSON
print("Descargando JSON con los datos de los cartones...")
files.download(json_path)

¡Generación completada! Iniciando descarga de archivos...
Descargando PDF...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargando JSON con los datos de los cartones...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>